(track1)=
# Track

In [1]:
import numpy as np
import scibmad as sb
sb.options.display_width = 100
sb.options.display_height = 30

## Basics

The `track` function can be used for all particle tracking, including spin tracking with
radiation, intra-beam scattering, etc. At the highest level, all one needs to do is provide
the beamline to track through and a matrix of size `n_particles x 6` of the initial particle
canonical phase space coordinates. Note that SciBmad uses the same six canonical coordinates
as in Bmad, in the order `x, px, y, py, z, pz`:

- `x`, `y`: the horizontal and vertical positions relative to the reference trajectory [m]
- `px`, `py`: the horizontal and vertical momenta divided by the reference momentum $P_0$
- `z` $= -\beta c (t - t_\mathrm{ref})$: the longitudinal position relative to the reference
  particle [m], positive when the particle is ahead of it ($\beta c$ is the particle's velocity,
  $t$ its arrival time and $t_\mathrm{ref}$ that of the reference particle)
- `pz` $= (P - P_0)/P_0$: the relative momentum deviation, also called $\delta$

The parameters of the beamline, including [deferred expressions](defexpr.md), are evaluated
once when `track` is called, and kept for all turns of that call. Changes to them take effect
at the next call of `track`. To have parameters change during tracking (for example a ramp),
use [time-dependent parameters](timedependent.md).

In [2]:
with sb.elements():
    qf = sb.Quadrupole(Kn1=0.36, L=0.5)
    bpm1 = sb.BPM()
    sf = sb.Sextupole(Kn2=1.2, L=0.2)
    d = sb.Drift(L=0.1)
    b = sb.SBend(L=1.2, angle=np.pi/132)
    qd = sb.Quadrupole(Kn1=-0.36, L=0.5)
    bpm2 = sb.BPM()
    sd = sb.Sextupole(Kn2=-1.2, L=0.2)

fodo = sb.Beamline([qf, bpm1, sf, b, d, qd, bpm2, sd, b, d],
                   species_ref=sb.Species("electron"), pc_ref=18e9)

v0 = np.array([[1e-3, 0, 0, 0, 0, 0],
               [1e-3, 0, 0, 0, 0, 1e-3]])
res = sb.track(fodo, v0=v0, n_turns=10)
res

TrackResult of 2 particles:
config:
 n_turns                         = 10
 save_every_n_turns              = 1
 in_body_coordinates             = false
 save_reference                  = false
 scalar_params                   = false
 rf_on                           = true
 ramp_particle_energy_without_rf = false
 ramp_update_each_particle       = false
 batch_start                     = 1
 verbose                         = false
 groupsize                       = 0
 use_cpu_multithreading          = false
 use_KA                          = false
 use_explicit_SIMD               = true

elapsed_time = 0.000145342 seconds

start = 1
turns = 0:1:10

s, index, name, kind    the saved locations: the end of the beamline (s = 4.0)
state                   particle state codes, e.g. state(turn=n, particle=i)
x, px, y, py, z, pz     the coordinates, e.g. x(turn=n, particle=i)
v                       all six coordinates, e.g. v(turn=n, particle=i)
data                    the internal, low-level 

The outputted structure is a `TrackResult`, which contains quite a lot of information
about the tracking. First, `config` is a `TrackConfig` structure that contains all the
configuration settings we may have provided to `track`, for example `n_turns`. For a
description of all settings, see the [`TrackConfig`](#track.config) docstring.

The integer value `start` specifies the beamline index of the element where tracking was commenced (at the entrance end). This can be changed by specifying the `start` keyword argument to `track`. The turns that the particle coordinates were saved at are also included (`res.turns`), with 0 being the initial coordinates.

The four vectors `s`, `index`, `name`, and `kind` specify each of the locations in the beamline where phase space coordinates were saved (the s-positions, and the containing elements' beamline indexes, names, and kinds). By default, the phase space coordinates are only saved at the end of the beamline, as in the above example. Further down on this page we discuss saving coordinates at different locations in the beamline.

:::{only} julia
Next, we have the `TrackColumn`s: `state`, `x`, `px`, `y`, `py`, `z`, `pz`, and `v`. All `TrackColumn`s store the particle data at every selected location and turn. If only one location is saved, then all `TrackColumn`s can be indexed using the syntax `x[turn=n, particle=i]` where `turn` and `particle` can be integers, ranges/vectors of indexes, or colons `:` which specifies all. Both `turn` and `particle` default to a colon `:`.
:::
:::{only} python
Next, we have the `TrackColumn`s: `state`, `x`, `px`, `y`, `py`, `z`, `pz`, and `v`. All
`TrackColumn`s store the particle data at every selected location and turn. If only one location is
saved, then all `TrackColumn`s are read with the syntax `x(turn=n, particle=i)` (in Julia
`x[turn=n, particle=i]`: Python has no keywords in indexing), where `turn` and `particle` can be
integers, ranges/lists of numbers, or `...` which specifies all. Both `turn` and `particle` default
to all. Particles are numbered from 1, as in Julia, and turn 0 holds the initial coordinates. The
values are NumPy arrays.
:::

In [3]:
print(res.x(turn=3))               # x of all particles after turn 3
print(res.x(turn=3, particle=1))   # x of particle 1
print(res.v(turn=3, particle=1))   # all six coordinates x, px, y, py, z, pz
res.x()                            # [particle, saved turn]

[-0.00050338  0.00030975]
-0.0005033780266342114
[-5.03378027e-04 -1.54376717e-04  5.27328303e-19 -9.69966185e-21
 -3.18775667e-05  0.00000000e+00]


array([[ 1.00000000e-03,  5.39822921e-04,  1.92958170e-05,
        -5.03378027e-04, -9.69846008e-04, -1.32810640e-03,
        -1.53826823e-03, -1.57694906e-03, -1.43984495e-03,
        -1.14220361e-03, -7.17151332e-04],
       [ 1.00000000e-03,  6.28310094e-04,  3.89333217e-04,
         3.09753359e-04,  3.98453705e-04,  6.45536946e-04,
         1.02341508e-03,  1.48985505e-03,  1.99265890e-03,
         2.47547740e-03,  2.88412160e-03]])

:::{only} julia
The saved turn `k` in `res.x[]` is the turn `res.turns[k]`: here the saved turns 1 to 11 are
the turns 0 to 10.
:::
:::{only} python
`res.x()` (without a location or keywords) is the array of all saved values, whose saved turn
`k` (a NumPy index, from 0) is the turn `res.turns[k]`: here the saved turns 0 to 10 are the
turns 0 to 10.
:::

The values of `state` are 8-bit integers that correspond to the following constants defined in
`BeamTracking`{pytext}` (sb.BeamTracking in Python)`:


```python
sb.BeamTracking.STATE_PREBORN                  # 0
sb.BeamTracking.STATE_ALIVE                    # 1
sb.BeamTracking.STATE_LOST                     # 2
sb.BeamTracking.STATE_LOST_NEG_X               # 3
sb.BeamTracking.STATE_LOST_POS_X               # 4
sb.BeamTracking.STATE_LOST_NEG_Y               # 5
sb.BeamTracking.STATE_LOST_POS_Y               # 6
sb.BeamTracking.STATE_LOST_PZ                  # 7
sb.BeamTracking.STATE_LOST_Z                   # 8
sb.BeamTracking.STATE_IMPLICIT_NONCONVERGENCE  # 9
```

Therefore, if a particle's state is {jl}`0x1`{py}`1`, that means it is alive and well! Once a
particle
is "lost" in tracking, its phase space coordinates are frozen at the point where it is lost.
`STATE_PREBORN` marks a location the particle has not reached yet (see
[Starting at a Different Location](#track.start)).

```{docstring} track
```

```{docstring} TrackResult
```

```{docstring} TrackColumn
```

## Saving Particle Data at Certain Locations

The particle data can also be saved at other locations given to `track` through the `at` keyword argument, which accepts a vector containing any combination of: 

- s-ranges as a tuple from start to stop, e.g. `(0., 1.23)` saves the coordinates for all integration steps between `s=0` and `s=1.23`
- Beamline element indexes as integers or index ranges (only at the beginning of the element).
  The end of the beamline is the index {jl}`length(fodo) + 1`{py}`len(fodo) + 1`, one more than the
  number of elements
- `LatElement`s themselves (only at the beginning of each placement of the element)
- Element names, as in [Searching by Name](#search:name), e.g. `"bpm*"`
- {jltext}`A colon `{jl}`:`{pytext}`An ellipsis `{py}`...`{pytext}` (or slice(None))` for every
  integration step

For example, to save the phase space coordinates at the entrance of the BPMs in the above line, use:

In [4]:
res = sb.track(fodo, v0=v0, n_turns=10, at=[bpm1, bpm2])
res

TrackResult of 2 particles:
config:
 n_turns                         = 10
 save_every_n_turns              = 1
 in_body_coordinates             = false
 save_reference                  = false
 scalar_params                   = false
 rf_on                           = true
 ramp_particle_energy_without_rf = false
 ramp_update_each_particle       = false
 batch_start                     = 1
 verbose                         = false
 groupsize                       = 0
 use_cpu_multithreading          = false
 use_KA                          = false
 use_explicit_SIMD               = true

elapsed_time = 7.1389e-5 seconds

start = 1
turns = 0:1:10

s, index, name, kind    the saved locations: bpm1 (s = 0.5), bpm2 (s = 2.5)
state                   particle state codes, e.g. state(location, turn=n, particle=i)
x, px, y, py, z, pz     the coordinates, e.g. x(location, turn=n, particle=i)
v                       all six coordinates, e.g. v(location, turn=n, particle=i)
data                   

When there is more than one location saved, indexing for all `TrackColumn`s requires specifying
which location(s) the data should be indexed at as the first indexing argument, using any of the
location indexing formats passable to `at` (element, element index, array of elements/indexes,
s-ranges, {jltext}`colon`{pytext}`...`).

In [5]:
res.x(bpm2, turn=10)          # x at bpm2 (the exit of qd) after turn 10

array([-0.00077511,  0.00239513])

If the end of a beamline is saved, then {jl}`end`{py}`-1` can be specified as the location for that
point. A location that is several places (e.g. `[[d]]` for every placement of element `d`,
{jl}`[qd, :]`{py}`(qd, ...)` for all integration steps through `qd`, an s-range) gives one value per
place, in the index after the particle. All of this is the exact same indexing scheme as for the
columns of `twiss` (see [Twiss](twiss.md)).

The s, and the beamline index, name and kind of the element of every saved location are in `res.s`, `res.index`, `res.name` and `res.kind`, so that for example the orbit through the beamline can be plotted:


```python
res = sb.track(fodo, v0=v0, n_turns=1, at=...)  # ... = Every integration step

import matplotlib.pyplot as plt
plt.plot(res.s, res.x(..., turn=1, particle=1))  # x vs. s for every integration step
```

(track.start)=
## Starting at a Different Location
The tracking can also start at the beginning of another element, given with `start`. The first
turn then goes from there to the end of the beamline (for example, `n_turns=1` tracks through
the rest of a linac), and the locations before the start are first reached in the second turn,
so they are `NaN` until then, also in turn 0 (with `state` equal to
`BeamTracking.STATE_PREBORN = 0x0`):

In [6]:
res = sb.track(fodo, v0=v0, n_turns=2, at=[qf, qd], start=qd)
res.x(qf, particle=1)       # [NaN, NaN (not reached in turns 0 and 1), x in turn 2]

array([       nan,        nan, 0.00131922])

Eventually when backwards tracking is supported by SciBmad, there will be an option to track backwards to the beginning of the beamline from any given `start`.

## Saving the Reference Time/Energy

The reference time and energy at the chosen locations can be saved too, with
{jl}`save_reference=True`{py}`save_reference=True`:
they are then `res.t_ref` (in seconds) and `res.beta_gamma_ref` (the reference energy as
$\beta\gamma$), one value per location and turn. This extra save makes the tracking slightly slower, so it is off by default:

In [7]:
res = sb.track(fodo, v0=v0, n_turns=2, at=[qf, qd], save_reference=True)
res.t_ref[qd]               # at the beginning of qd, for every saved turn

array([0.00000000e+00, 6.67128191e-09, 2.00138457e-08])

## Spin Tracking

To do spin tracking, simply set {jl}`spin=True`{py}`spin=True`. This will initialize all particles
with their
own spin [quaternion](https://en.wikipedia.org/wiki/Quaternion), each of which will be
tracked along with the particle. A quaternion is essentially a rotation matrix - by tracking
quaternions instead of spin 3-vectors, we can track one particle once and then observe the
spin dynamics for any initial spin direction for that particle.

In [8]:
v0 = [[1e-3, 0, 0, 0, 0, 1e-3]]
res = sb.track(fodo, v0=v0, n_turns=10, spin=True)
res

TrackResult of 1 particle:
config:
 n_turns                         = 10
 save_every_n_turns              = 1
 in_body_coordinates             = false
 save_reference                  = false
 scalar_params                   = false
 rf_on                           = true
 ramp_particle_energy_without_rf = false
 ramp_update_each_particle       = false
 batch_start                     = 1
 verbose                         = false
 groupsize                       = 0
 use_cpu_multithreading          = false
 use_KA                          = false
 use_explicit_SIMD               = true

elapsed_time = 0.000133326 seconds

start = 1
turns = 0:1:10

s, index, name, kind    the saved locations: the end of the beamline (s = 4.0)
state                   particle state codes, e.g. state(turn=n, particle=i)
x, px, y, py, z, pz     the coordinates, e.g. x(turn=n, particle=i)
v                       all six coordinates, e.g. v(turn=n, particle=i)
q                       spin quaternions (track_s

The spin quaternions are then `res.q`, for example
{jl}`res.q[turn=5, particle=1]`{py}`res.q(turn=5, particle=1)` (its four
components): the rotation of the spin from the start of the tracking through turn 5.

To compute a particle's spin 3-vector at each turn given some initial spin direction, we can
use the post-processing function `track_spin`:

In [9]:
s0 = [0, 1, 0]  # initial vertical spin vector
s = sb.track_spin(res.q, s0)
s

TrackColumn `spin` (read it with `spin(turn=n, particle=i)`)

`s` is read just like `res.q`, but with the three components of the spin vector:

In [10]:
s(turn=10, particle=1)      # the spin of particle 1 after turn 10

array([1.17470752e-16, 1.00000000e+00, 3.53527619e-16])

If you'd like to initialize different initial spin directions for each particle (perhaps
along the invariant spin field), give the initial spins as a matrix of size `n_particles x 3`.

(track.config)=
## Configuration Settings

While configuration settings can be set as keyword arguments at the level of `track`, a
`TrackConfig` structure may be provided instead, to the keyword argument `config`. All of
these settings are described below in the docstring for `TrackConfig`.

```{docstring} TrackConfig
```

## The Bunch Struct

While initial particle phase space coordinates, quaternions, macroparticle weights, and more
can be set at the level of `track`, these are ultimately used to construct a `Bunch`, which
may alternatively be provided to `track` with the keyword argument `bunch`. A `Bunch`
contains all information about a particle bunch, including its species, current reference
time `t_ref` (used to evaluate [time-dependent parameters](timedependent.md)), and current
reference energy `p_over_q_ref`. All `Bunch` properties are described in the below
docstring.

If we'd like to continue tracking more turns with a bunch used in previous tracking, we can simply provide this bunch to `track`:

In [11]:
res = sb.track(fodo, v0=v0, n_turns=5)

# track 5 more turns:
res2 = sb.track(fodo, bunch=res.bunch, n_turns=5)
res2

TrackResult of 1 particle:
config:
 n_turns                         = 5
 save_every_n_turns              = 1
 in_body_coordinates             = false
 save_reference                  = false
 scalar_params                   = false
 rf_on                           = true
 ramp_particle_energy_without_rf = false
 ramp_update_each_particle       = false
 batch_start                     = 1
 verbose                         = false
 groupsize                       = 0
 use_cpu_multithreading          = false
 use_KA                          = false
 use_explicit_SIMD               = true

elapsed_time = 3.4501e-5 seconds

start = 1
turns = [0, 1, 2, 3, 4, 5]

s, index, name, kind   the saved locations: the end of the beamline (s = 4.0)
state                  particle state codes, e.g. state(turn=n, particle=i)
x, px, y, py, z, pz    the coordinates, e.g. x(turn=n, particle=i)
v                      all six coordinates, e.g. v(turn=n, particle=i)
data                   the internal, low-lev

The `Bunch` is tracked in place, so afterwards `res.bunch` and `res2.bunch` are the same
`Bunch`, after all 10 turns. The saved coordinates in `res` and `res2` are not changed.

```{docstring} Bunch
```

## Tracking a Different Species

By default, the particles tracked are of the reference species of the beamline. To track a
different species, give it to `track` with `species` (or give `track` a `Bunch` of that
species). Just like in a real accelerator,
the magnets keep the same fields, and each particle responds to them according to its own
charge, mass, and energy. For example, muons with the same momentum as the design electrons
follow the same orbit, but they are slower, so they fall behind (their `z` changes more):

In [12]:
res = sb.track(fodo, v0=v0, species=sb.Species("muon"), n_turns=10)
res.v(turn=10, particle=1)

Tracking Species("muon", -1.0, 1.056583755e8, 0.5, -2.00233184123, -2.80271738128469e-7, 0, AtomicAndPhysicalConstants.Kind.LEPTON) through a Beamline with reference species Species("electron", -1.0, 510998.95069, 0.5, -2.00231930436092, -5.795094307747843e-5, 0, AtomicAndPhysicalConstants.Kind.LEPTON): the fields of the Beamline are held fixed


array([ 2.88412160e-03,  3.38365793e-04,  4.98326946e-18, -7.46075592e-20,
       -1.19327262e-03,  1.00000000e-03])

A particle with the opposite charge, such as a positron in an electron ring, is bent the other
way by the dipoles and is quickly lost, just as it would be in the real ring.


(track.callbacks)=
## Callbacks

The `Bunch` struct may also contain a tuple of functions `callbacks`. These functions are
executed **every integration step**, and allow users to inject their own code into the
tracking if desired. An element with `n_slices` slices of `n_steps` steps each (see
[Integration Steps and Slices](#track.steps)) executes them `n_slices*n_steps` times, and
elements without integration steps (patches, maps, and elements tracked with `SaganCavity`) once. For example, `twiss` uses callbacks to compute its results at every integration step, inside of elements.

:::{only} python
Callbacks run inside SciBmad's compiled tracking kernels (also on GPUs), for every particle and
integration step, so they must be Julia functions. From Python, define one with
`sb.julia.seval("function mycallback(i, coords, p) ... end")`, which returns the Julia function.
This is the one part of SciBmad that needs Julia code.
:::

Callbacks are required to have the function signature defined below

```julia
function mycallback(i, coords::BeamTracking.Coords, p::TrackPoint)
  # Do stuff...
  return
end
```

In the tracking, each callback is executed and provided the arguments shown above in the
callback signature. These are:

- `i`: Particle index
- `coords`: A `BeamTracking.Coords` struct, containing the fields:
  - `coords.state`: particle state vector of length `n_particles`
  - `coords.v`: particle phase space matrix of size `n_particle x 6`
  - `coords.q`: `None` if no spin tracking, else, particle spin quaternion matrix of size `n_particle x 4`
  - `coords.weight`: `None` if uniform macroparticle weights, else weights vector of length `n_particles`
- `p`: A `TrackPoint`, the point of the tracking, with the properties:
  - `p.s`: Current s-position from the beginning of the beamline
  - `p.t_ref`: Current reference time of the bunch **from the beginning of tracking**
  - `p.step`: Current integration step from the beginning of the beamline (the steps of the
    elements counted as above, the last one of an element at its exit). This is how `track` finds
    the locations of `at`
  - `p.beta_gamma_ref`: Current reference Lorentz $\beta\gamma$ of the bunch
  - `p.last_ds_step`: Last integration step length before executing this callback
  - `p.last_g`: Last integration coordinate system curvature as a tuple `(gx, gy)` in lab coordinates before executing this callback

The functions `transforms_out!` and `transforms_in!` can be applied inside the callback to `coords` in order to transform the particle coordinates out of the element integration body frame and back into the laboratory frame. This can be useful if, for example, an element is misaligned, but you want to see the phase space coordinates in the laboratory frame. The transformations may be called by
simply executing `transforms_out!(i, coords, p)` or `transforms_in!(i, coords, p)`

Any of these quantities may be used in the callback. Also, as long as callbacks are written
to be GPU compatible, they will work on the GPU. For example, this callback prints the $x$
coordinate of each particle on the GPU in laboratory coordinates, each integration step:

```julia
using CUDA

function mycallback(i, coords, p)
  transforms_out!(i, coords, p) # Transform out of body integration coordinates
  CUDA.@cuprintln("Hi! I'm particle $i at $(p.t_ref) and my x-coordinate is $(coords.v[i,1])")
  transforms_in!(i, coords, p) # Transform back into body integration coordinates
  return
end
```

```{docstring} TrackPoint
```

## Coordinates Number Type
By default, particle phase space (and quaternion) coordinates are 64-bit floats. However, there may be cases where the often-significant performance gains of using 32-bit floats outweigh the cost of lower precision. For example, almost every neural network implementation in major machine learning libraries defaults to 32-bit precision, because it is "good enough". Furthermore, some GPUs (such as Apple Metal) _only_ support 32-bit floats. 

SciBmad tracking is also fully compatible with end-to-end 32-bit float tracking. To use this, simply initialize the particle phase space coordinates (and optionally spin quaternions) using 32-bit floats,

In [13]:
v0 = np.zeros((1, 6), dtype=np.float32)  # 1 32-bit particle
res = sb.track(fodo, v0=v0)
res

TrackResult of 1 particle:
config:
 n_turns                         = 1
 save_every_n_turns              = 1
 in_body_coordinates             = false
 save_reference                  = false
 scalar_params                   = false
 rf_on                           = true
 ramp_particle_energy_without_rf = false
 ramp_update_each_particle       = false
 batch_start                     = 1
 verbose                         = false
 groupsize                       = 0
 use_cpu_multithreading          = false
 use_KA                          = false
 use_explicit_SIMD               = true

elapsed_time = 7.1582e-5 seconds

start = 1
turns = [0, 1]

s, index, name, kind   the saved locations: the end of the beamline (s = 4.0)
state                  particle state codes, e.g. state(turn=n, particle=i)
x, px, y, py, z, pz    the coordinates, e.g. x(turn=n, particle=i)
v                      all six coordinates, e.g. v(turn=n, particle=i)
data                   the internal, low-level data arra

:::{note}
When 32-bit floats are used for the particle coordinates, _all_ numbers in the tracking (e.g. magnet strengths, lengths, and time-dependent parameters) are 32-bit floats too.
:::

(cpuparallel)=
## CPU Parallelization

On the CPU, all tracking will automatically be parallelized using explicitly-programmed
single instruction, multiple data (SIMD) instructions if your hardware supports it. To
disable explicit SIMD (and rely on potential compiler auto-vectorization), run `track` with
the keyword argument `use_explicit_SIMD=False`.

### Multithreading

:::{only} julia
SciBmad tracking is also compatible with CPU multithreading. To do so, Julia must be started
with threads. In a terminal:

```
$ julia --threads=auto
```

where `auto` will infer a number of threads to use based on your system, or alternatively an
integer number of threads may be specified. To check in a Julia session that multiple
threads are available, run the command
:::
:::{only} python
SciBmad tracking is also compatible with CPU multithreading. To do so, Julia must be started
with threads: call `sb.init(threads="auto")` before anything else of SciBmad is used (or set
the environment variable `SCIBMAD_THREADS=auto`, or run `python -m scibmad config --threads auto`
once), where `auto` will infer a number of threads to use based on your system, or
alternatively an integer number of threads may be given. To check that multiple threads are
available, run the command
:::


```python
print(sb.julia.Threads.nthreads())
```

If this number is > 1, then you are ready to do multithreaded tracking. Simply set the
`track` configuration setting {jl}`use_cpu_multithreading=True`{py}`use_cpu_multithreading=True`.

:::{note}
There is a cost to launching threads on the CPU. Using multithreading with a small number of
particles may actually slow down your tracking. We recommend checking with a small number of
turns first to see if multithreading can improve your tracking performance, before doing a
full long-term run.
:::

### Multiple Processes/Nodes

(Documentation in development, see
[Distributed](https://docs.julialang.org/en/v1/manual/distributed-computing/) and
[SlurmClusterManager](https://github.com/JuliaParallel/SlurmClusterManager.jl) in the
meantime)

(gpuparallel)=
## GPU Parallelization

**All SciBmad tracking is fully GPU parallelized and branchless.** To do particle tracking
on the GPU, either initialize a `Bunch` with the corresponding device `GPUArray`, or provide
the `GPUArray` to
`track`{pytext}` (in Python, sb.device("cuda").asarray(...) copies a NumPy array to a CUDA GPU; also "metal", "amdgpu" and "oneapi"; the first use installs the Julia package of that GPU kind, once)`.
For example, to do GPU-parallelized spin tracking on a CUDA GPU,


```python
gpu = sb.device("cuda")

n_particles = 100000

# Random initial phase space coordinates:
v0 = gpu.asarray(np.random.rand(n_particles, 6) * 1e-5)

res = sb.track(fodo, v0=v0, spin=True, n_turns=100)
```

:::{note}
Julia is a just-in-time (JIT) compiled language, meaning that many functions (including the
tracking kernels in SciBmad) are only compiled once they are called with specific types
(e.g., a `GPUArray`). We try to precompile for as many cases as possible, but precompilation
for `GPUArray`s is not possible. As such, the first "turn" in tracking will have a latency
in order for the JIT compiler to compile SciBmad's GPU tracking kernels.
:::

Apple Metal GPUs only support 32-bit floats, so on a Mac use `Float32` coordinates; all of
the tracking, including spin tracking, will then be done in 32-bit:


```python
gpu = sb.device("metal")
v0 = gpu.asarray(np.random.rand(n_particles, 6) * 1e-5, dtype=np.float32)
res = sb.track(fodo, v0=v0, spin=True, n_turns=100)
```

### Multiple GPUs

(Documentation in development, see
[CUDA: Multiple GPUs](https://cuda.juliagpu.org/stable/usage/multigpu/#Multiple-GPUs) in the
meantime)